<a href="https://colab.research.google.com/github/iqour/salary-from-description/blob/data-load-pipeline/Pet_Adoption_Speed_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


ValueError: mount failed

In [ ]:
from google.colab import files
files.upload()


In [ ]:
!pip install -q kaggle

# Create a hidden .kaggle directory and move the uploaded JSON there
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

# Set permissions so it works correctly
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle competitions download -c petfinder-adoption-prediction


In [ ]:
!unzip -q petfinder-adoption-prediction.zip -d petfinder_data


In [ ]:
import pandas as pd
import re

# Load training metadata (main file)
df = pd.read_csv('petfinder_data/train/train.csv')

# Quick overview
print("Dataset shape:", df.shape)
print("\nColumns and types:")
print(df.info())

print("\nSample rows:")
print(df.head())

print("\nTarget distribution (AdoptionSpeed):")
print(df['AdoptionSpeed'].value_counts(normalize=True))



**Section 2: Data Cleaning & feature Selection**


---



In [ ]:
df_clean = df.drop(columns = ['Name', 'RescuerID', 'PetID'])

# fill missing descriptions with empty string
df_clean['Description'] = df_clean['Description'].fillna('')

#standardise text formats for Description
def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^\w\s]', '', text)
  text = re.sub(r'\d+', '', text)
  return text
df_clean['clean_desc'] = df_clean['Description'].apply(clean_text)

# check categorical data that needs encoding
categorical_cols = ['Type', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3',
                    'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized',
                    'Health', 'State']
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
for col in categorical_cols:
  df_clean[col] = label_encoder.fit_transform(df_clean[col])

print(df_clean[['clean_desc']].head())
print("\nSample data after cleaning:")
print(df_clean.head())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize vectorizer
tfidf = TfidfVectorizer(max_features=500, stop_words='english')

# Fit and transform the text column
tfidf_matrix = tfidf.fit_transform(df_clean['clean_desc'])

# Convert sparse matrix to dense DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

# Check result
print("TF-IDF shape:", tfidf_df.shape)
print(tfidf_df.head())


In [ ]:
from sklearn.model_selection import train_test_split

# 1. Select structured features
# Drop text and original target columns
X_structured = df_clean.drop(columns=['Description', 'clean_desc', 'AdoptionSpeed'])

# 2. Combine structured + text features
import numpy as np
X_final = pd.concat([X_structured.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)

# 3. Target variable
y = df_clean['AdoptionSpeed'].astype(int)  # Convert to int if it's not

# 4. Split into train/val sets
X_train, X_val, y_train, y_val = train_test_split(X_final, y, test_size=0.2, random_state=42)

# Check shapes
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)


In [ ]:
!pip install xgboost --quiet

from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize the model
xgb = XGBClassifier(
    objective='multi:softmax',  # for multi-class classification
    num_class=5,                # 5 adoption speed levels (0-4)
    eval_metric='mlogloss',    # multi-class loss
    use_label_encoder=False,
    random_state=42
)

# 2. Train the model
xgb.fit(X_train, y_train)

# 3. Make predictions
y_pred = xgb.predict(X_val)

# 4. Evaluate
print("Classification Report:")
print(classification_report(y_val, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(zip(np.unique(y_train), class_weights))

# Map sample weights for each training sample
sample_weights = y_train.map(class_weights_dict)

# Initialize model
xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    eval_metric='mlogloss',
    random_state=42,
)

# Train with sample weights
xgb.fit(X_train, y_train, sample_weight=sample_weights)

# Predict and evaluate
y_pred = xgb.predict(X_val)

print("Classification Report:")
print(classification_report(y_val, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import matplotlib.pyplot as plt

# 1. Compute class weights (you may or may not use this depending on approach)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(zip(np.unique(y_train), class_weights))
print("Class weights:", class_weights_dict)

# 2. Initialize RandomForest WITHOUT sample_weight, using class_weight balanced
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',  # balanced weights internally
    random_state=42,
    n_jobs=-1
)

# 3. Train model on training data
rf.fit(X_train, y_train)  # no sample_weight passed here

# 4. Predict on validation data
y_pred = rf.predict(X_val)

# 5. Evaluate
print("Classification Report:")
print(classification_report(y_val, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

# 6. Feature importances plot (optional)
importances = rf.feature_importances_
indices = np.argsort(importances)[-20:]  # top 20 features

plt.figure(figsize=(10,6))
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [X_train.columns[i] for i in indices])
plt.title("Top 20 Feature Importances in RandomForest")
plt.show()


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [8, 10, 12],
    'min_samples_split': [2, 5]
}

rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='f1_weighted',   # more robust than just accuracy
    cv=3,
    verbose=1
)

grid_search.fit(X_train, y_train)
print("Best Params:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Use the best model
best_rf = grid_search.best_estimator_

# Predict on validation set
y_pred = best_rf.predict(X_val)

# Evaluate
print("Classification Report:")
print(classification_report(y_val, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))
